# 💼 Real-World Quantitative Fintech Interview Cases (NumPy Practice)

Welcome to your **NumPy Quantitative Fintech Interview Practice Notebook**!

In technical interviews for Quantitative Analyst, Risk Engineer, and Machine Learning roles at top firms (Citadel, Stripe, Robinhood, Jane Street, AmEx), you will face mathematical and algorithmic problems. Always follow the **5-Step Reverse Engineering Framework**:

```
┌──────────────────────────────┐     ┌──────────────────────────────┐     ┌──────────────────────────────┐
│ 1. Mathematical Contract     │ ──> │ 2. Raw Matrix & Memory Audit │ ──> │ 3. Zero-Loop Vectorization   │
│ (Equations, Shapes, Dtypes)  │     │ (Shapes, Strides, NaNs, Infs)│     │ (@, Ufuncs, Sliding Windows) │
└──────────────────────────────┘     └──────────────────────────────┘     └──────────────────────────────┘
```

---
### 📁 Available Datasets in `./data/`
- `data/multi_asset_returns.npy` (252 Trading Days x 50 Asset Return Matrix, float64)
- `data/portfolio_covariance_matrix.npy` (50 x 50 Symmetric Empirical Covariance Matrix, float64)
- `data/l2_orderbook_depth.npz` (5,000 High-Frequency L2 Snapshots with 5 Bids and 5 Asks depth levels)
- `data/loan_risk_features.npy` (5,000 Applicants x 8 Standardized Underwriting Risk Features)
- `data/customers.csv` (Customer metadata, KYC status, balances, and risk tiers)

In [ ]:
# Run this setup cell first to initialize the environment and load real datasets
import numpy as np
import pandas as pd
import os

DATA_DIR = 'data'

daily_returns = np.load(os.path.join(DATA_DIR, 'multi_asset_returns.npy'))
covariance_matrix = np.load(os.path.join(DATA_DIR, 'portfolio_covariance_matrix.npy'))
orderbook_data = np.load(os.path.join(DATA_DIR, 'l2_orderbook_depth.npz'))
loan_features = np.load(os.path.join(DATA_DIR, 'loan_risk_features.npy'))

print("✅ Environment ready! Loaded quantitative datasets:")
print(f"   • Multi-Asset Daily Returns  : Shape {daily_returns.shape} (float64)")
print(f"   • Portfolio Covariance Matrix: Shape {covariance_matrix.shape} (float64)")
print(f"   • L2 Orderbook Asks          : Shape {orderbook_data['asks'].shape} (float64)")
print(f"   • Loan Underwriting Features : Shape {loan_features.shape} (float64)")

---
## 🏢 Case 1 (Quantitative Risk): Annualized Portfolio Risk & Sharpe Ratio Optimization

### 📌 Business Scenario
> **Stakeholder:** Head of Quantitative Risk & Portfolio Management  
> *"We are rebalancing our multi-asset fund across 50 equities. I need you to compute the risk-adjusted return profile for an equal-weighted portfolio. Specifically, calculate the annualized expected portfolio return, annualized portfolio volatility, and the annualized Sharpe ratio assuming a risk-free rate of 2.0% ($R_f = 0.02$). This must be completely vectorized with zero Python loops."*

### 🎯 Step 1: Target Mathematical Contract
- Equal Weights: $w_i = \frac{1}{50}$ such that $\sum w_i = 1.0$ (Vector $w \in \mathbb{R}^{50}$)
- Asset Mean Returns: $\mu = \text{mean}(\text{daily\_returns}, \text{axis}=0)$
- Annualized Expected Return: $R_p = (w \cdot \mu) \times 252$
- Annualized Portfolio Variance: $\sigma_p^2 = (w^T \Sigma w) \times 252$
- Annualized Volatility: $\sigma_p = \sqrt{\sigma_p^2}$
- Annualized Sharpe Ratio: $\text{Sharpe} = \frac{R_p - R_f}{\sigma_p}$

### 🔍 Step 2: Raw Data & Matrix Audit
- `daily_returns`: Shape `(252, 50)`
- `covariance_matrix`: Shape `(50, 50)`

### 🛠️ Step 3: Your Solution (Write Your Code Below)

In [ ]:
# Case 1: Write your vectorized solution below
# Inputs available: daily_returns, covariance_matrix
risk_free_rate = 0.02

# TODO: Calculate annualized_return, annualized_volatility, sharpe_ratio




---
## 🏢 Case 2 (Hedge Fund Telemetry): High-Water Mark & Maximum Drawdown (MDD)

### 📌 Business Scenario
> **Stakeholder:** Chief Investment Officer (CIO)  
> *"Our risk committee requires a maximum drawdown audit on our primary equity strategy. Reconstruct the cumulative price series starting at an index of $100.0 from daily returns of Asset 0 (`daily_returns[:, 0]`). Track the running peak (High-Water Mark), calculate the daily percentage drawdowns, and find the worst peak-to-trough crash (Maximum Drawdown) and the exact day it hit bottom."*

### 🎯 Step 1: Target Mathematical Contract
- Price Series: $S_t = 100.0 \times \prod_{i=1}^t (1 + r_i)$ via `np.cumprod`
- High-Water Mark (Running Peak): $M_t = \max_{1 \le \tau \le t}(S_\tau)$ via `np.maximum.accumulate`
- Percentage Drawdown: $D_t = \frac{S_t - M_t}{M_t}$
- Maximum Drawdown: $\text{MDD} = \min(D_t)$
- Trough Day Index: $t_{\text{trough}} = \text{argmin}(D_t)$

### 🔍 Step 2: Raw Data & Matrix Audit
- Asset Returns: `asset_returns = daily_returns[:, 0]` (Shape `(252,)`)

### 🛠️ Step 3: Your Solution (Write Your Code Below)

In [ ]:
# Case 2: Write your vectorized solution below
# Input available: daily_returns[:, 0]
asset_returns = daily_returns[:, 0]

# TODO: Calculate cumulative_prices, running_peaks, drawdowns, max_drawdown_pct, trough_day_index




---
## 🏢 Case 3 (Algorithmic Trading): Zero-Copy Rolling Bollinger Bands via Stride Tricks

### 📌 Business Scenario
> **Stakeholder:** Lead Quantitative Market-Making Trader  
> *"We are building a mean-reversion trading strategy. We need a 20-day rolling moving average and upper/lower Bollinger Bands ($\mu \pm 2\sigma$) over our cumulative price stream. Because this runs on low-latency tick engines, you must use zero-copy memory windowing (`sliding_window_view`) with zero Python loops and zero duplicate RAM allocation."*

### 🎯 Step 1: Target Mathematical Contract
- Window Size: $W = 20$
- Overlapping Sub-Windows: Shape `(252 - 20 + 1, 20) = (233, 20)` via `sliding_window_view`
- Rolling Mean: $\mu_t = \text{mean}(\text{windows}, \text{axis}=1)$ (Shape `(233,)`)
- Rolling Volatility: $\sigma_t = \text{std}(\text{windows}, \text{axis}=1)$ (Shape `(233,)`)
- Upper Band: $U_t = \mu_t + 2\sigma_t$
- Lower Band: $L_t = \mu_t - 2\sigma_t$

### 🔍 Step 2: Raw Data & Matrix Audit
- Prices Input: Reconstructed cumulative price series from Case 2 (Shape `(252,)`)

### 🛠️ Step 3: Your Solution (Write Your Code Below)

In [ ]:
# Case 3: Write your zero-copy sliding window solution below
from numpy.lib.stride_tricks import sliding_window_view

# TODO: Generate rolling 20-day windows, rolling_means, upper_band, lower_band




---
## 🏢 Case 4 (Smart Order Routing): L2 Orderbook Depth Matching & Execution VWAP

### 📌 Business Scenario
> **Stakeholder:** Head of Execution Algorithms & Brokerage Infrastructure  
> *"An institutional client wants to execute an immediate market buy order of 1,500 shares of AAPL. Inspect the current top 5 ask levels from our L2 orderbook feed (`orderbook_data['asks'][0]`), where each row is `[Ask Price, Available Size]` sorted from best price to deepest price. Calculate how many shares get filled at each depth level, total dollar expenditure, and the effective Volume-Weighted Average Price (VWAP)."*

### 🎯 Step 1: Target Mathematical Contract
- Target Order Quantity: $Q_{\text{target}} = 1,500.0$ shares
- Ask Ladder: Matrix of shape `(5, 2)` containing `[Price, Size]`
- Cumulative Available Shares: $C_i = \sum_{k=1}^i \text{Size}_k$
- Shares Filled per Level: $F_i = \text{clip}(Q_{\text{target}} - C_{i-1}, 0, \text{Size}_i)$
- Total Dollar Spend: $\text{Spend} = \sum (F_i \times \text{Price}_i)$
- Execution VWAP: $\text{VWAP} = \frac{\text{Spend}}{Q_{\text{target}}}$

### 🔍 Step 2: Raw Data & Matrix Audit
- `orderbook_data['asks'][0]`: Shape `(5, 2)` float64

### 🛠️ Step 3: Your Solution (Write Your Code Below)

In [ ]:
# Case 4: Write your vectorized orderbook execution below
ask_ladder = orderbook_data['asks'][0]  # Shape (5, 2) -> [Price, Size]
target_order_shares = 1500.0

# TODO: Calculate shares_filled_per_level, total_spend_usd, execution_vwap




---
## 🏢 Case 5 (Credit Underwriting & Machine Learning): Vectorized Default Risk Scoring

### 📌 Business Scenario
> **Stakeholder:** Head of Credit Underwriting & Data Science  
> *"We are deploying a linear credit default scoring model across 5,000 credit card applicants in `loan_features` (Shape: `(5000, 8)`). First, standardize all 8 feature columns to Z-Scores ($Z = \frac{X - \mu}{\sigma}$). Next, compute the logistic default probabilities $P = \sigma(Z W + b)$ via matrix multiplication `@`. Finally, flag all applicants with $P(\text{Default}) > 0.65$ as high-risk and report the total count."*

### 🎯 Step 1: Target Mathematical Contract
- Raw Features Matrix: $X \in \mathbb{R}^{5000 \times 8}$
- Feature Means & Stds: $\mu = \text{mean}(X, \text{axis}=0)$, $\sigma = \text{std}(X, \text{axis}=0)$
- Z-Score Normalization: $Z = \frac{X - \mu}{\sigma}$ via column broadcasting
- Model Weights: $W = [0.45, -0.20, 0.35, 0.15, -0.40, 0.30, 0.50, 0.60]^T$ ($W \in \mathbb{R}^{8}$), Bias $b = -0.50$
- Linear Logits: $z = Z W + b$ (Vector $z \in \mathbb{R}^{5000}$ via `@`)
- Default Probability: $P(\text{Default}) = \frac{1}{1 + e^{-z}}$
- High-Risk Mask: $P > 0.65$

### 🔍 Step 2: Raw Data & Matrix Audit
- `loan_features`: Shape `(5000, 8)` float64

### 🛠️ Step 3: Your Solution (Write Your Code Below)

In [ ]:
# Case 5: Write your vectorized risk scoring solution below
model_weights = np.array([0.45, -0.20, 0.35, 0.15, -0.40, 0.30, 0.50, 0.60]) # Shape (8,)
model_bias = -0.50

# TODO: Compute standardized_Z, default_probabilities, high_risk_flags, total_high_risk_count


